In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/assignment/employees (1).csv",
    header=True,inferSchema=True
)

In [0]:
df1 = spark.read.csv("/Volumes/workspace/default/assignment/sales.csv", header = True)

### Reading the multiple file same time

In [0]:
file = {
    "employee":"/Volumes/workspace/default/assignment/employees (1).csv",
    "sales":"/Volumes/workspace/default/assignment/sales.csv"
}

df = {}

for name, path in file.items():
    df[name]=spark.read.csv(
        path,
        header=True,
        inferSchema=True
    )

In [0]:
df["employee"].show()

+----------+-----------+----------------+------+---------+-----------+---+------+---------------+
|EmployeeID|       Name|      Department|Salary|     City|JoiningDate|Age|Gender|Education_Level|
+----------+-----------+----------------+------+---------+-----------+---+------+---------------+
|         1| Employee_1|              HR| 31556|    Delhi| 2022-03-04| 36|  Male|         Master|
|         2| Employee_2|              HR|167964|Bangalore| 2023-04-05| 58|Female|       Bachelor|
|         3| Employee_3|              HR| 82314|   Mumbai| 2015-05-03| 53|  Male|         Master|
|         4| Employee_4|Customer Support|134974|   Mumbai| 2023-01-11| 49|Female|       Bachelor|
|         5| Employee_5|         Finance|135785|     Pune| 2023-07-06| 38|  Male|         Master|
|         6| Employee_6|      Operations| 51793|Bangalore| 2023-07-25| 45|  Male|            PhD|
|         7| Employee_7|      Operations|183263|    Delhi| 2024-07-03| 23|Female|       Bachelor|
|         8| Employe

### Inner join

In [0]:
df["employee"].join(
    df["sales"],
    "EmployeeID","inner"
).select("EmployeeID","Product","Quantity","SalesAmount").display()

EmployeeID,Product,Quantity,SalesAmount
1,Analytics Package,11,380881
1,Analytics Package,2,16488
1,Support Package,4,448799
1,Analytics Package,3,200869
1,Cloud Service,19,57969
1,Subscription,5,87577
1,Consulting,11,96332
1,Support Package,8,353930
1,Analytics Package,13,165863
1,Subscription,6,387117


### Left join

In [0]:
df["employee"].join(
    df["sales"],
    "EmployeeID","left"
).select("EmployeeID","Product","Quantity","SalesAmount").display()

EmployeeID,Product,Quantity,SalesAmount
1,Subscription,6,387117
2,Laptop,19,346295
4,Laptop,13,304772
5,Consulting,9,123616
7,Laptop,17,224483
9,Support Package,18,175984
11,Support Package,4,222287
12,Cloud Service,8,75456
13,Laptop,7,148643
14,Laptop,5,344409


### Left Anti join

In [0]:
df["employee"].join(
    df["sales"],
    "EmployeeID","left_anti"
).select("EmployeeID","Department").display()

EmployeeID,Department
3,HR
6,Operations
8,HR
10,HR
16,Procurement
17,Analytics
20,Marketing
22,Finance
23,HR
24,Procurement


### Right Join

In [0]:
df["employee"].join(
    df["sales"],
    "EmployeeID","right"
).select("EmployeeID","SalesAmount").display()

EmployeeID,SalesAmount
1,380881
1,16488
1,448799
1,200869
1,57969
1,87577
1,96332
1,353930
1,165863
1,387117


### Outer Join

In [0]:
df["employee"].join(
    df["sales"],
    "EmployeeID","outer"
).select("EmployeeID","SalesAmount","Department").display()

EmployeeID,SalesAmount,Department
12,300008,Analytics
12,466444,Analytics
12,348297,Analytics
12,95282,Analytics
12,324110,Analytics
12,212509,Analytics
12,395987,Analytics
12,182139,Analytics
12,454962,Analytics
12,75456,Analytics


### Left Semi Join

In [0]:
df["employee"].join(
    df["sales"],
    "EmployeeID","left_semi"
).select("EmployeeID","Department").display()

EmployeeID,Department
1,HR
2,HR
4,Customer Support
5,Finance
7,Operations
9,HR
11,Operations
12,Analytics
13,IT
14,Sales


### Cross join

In [0]:
df["employee"].join(
    df["sales"],
    "EmployeeID","cross"
).select("EmployeeID","Department","SalesAmount").display()

EmployeeID,Department,SalesAmount
1,HR,380881
1,HR,16488
1,HR,448799
1,HR,200869
1,HR,57969
1,HR,87577
1,HR,96332
1,HR,353930
1,HR,165863
1,HR,387117


### BroadCast

In [0]:
from pyspark.sql.functions import broadcast
df["employee"].join(
    broadcast(df["sales"]),
    "EmployeeID","inner"
).select("EmployeeID","Department","SalesAmount").display()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5030535202309021>, line 3
      1 from pyspark.sql.functions import broadcast
----> 3 df["employee"].join(
      4     broadcast(df["sales"]),
      5     "EmployeeID","outer"
      6 ).select("EmployeeID","Department","SalesAmount").display()

NameError: name 'df' is not defined

## Handling Nulls

In [0]:
df = spark.table("workspace.default.messy_customers_large")
display(df)

CustomerID,Name,Age,City,Salary,JoinDate
1,Kiran,44,bangalore,NULL,2023-04-01
2,Meena,24,BANGALORE,27891,2023-09-04
3,John,null,chennai,not_available,2023-01-04
4,Meena,null,Pune,40297,2023-05-18
5,Asha,39,Delhi,NULL,2023-08-28
6,Sneha,null,delhi,not_available,2023-01-10
7,Manoj,abc,Delhi,37696,2023-09-09
8,Rahul,21,Chennai,NULL,2023-04-27
9,Priya,abc,bangalore,NULL,2023-09-16
10,Ravi,abc,Bangalore,37334,2023-04-22


### 1) Find nulls

In [0]:
from pyspark.sql.functions import col

df.filter(
    col("Age").isNull()
).show()

+----------+------+----+---------+-------------+----------+
|CustomerID|  Name| Age|     City|       Salary|  JoinDate|
+----------+------+----+---------+-------------+----------+
|         3|  John|NULL|  chennai|not_available|2023-01-04|
|         4| Meena|NULL|     Pune|        40297|2023-05-18|
|         6| Sneha|NULL|    delhi|not_available|2023-01-10|
|        13| Suraj|NULL|    Delhi|         NULL|2023-06-28|
|        16| Rahul|NULL|Hyderabad|not_available|2023-04-18|
|        21| Kumar|NULL|      blr|         NULL|2023-04-20|
|        22| Divya|NULL|  chennai|        26094|2023-07-03|
|        23|  Sana|NULL|    Delhi|         NULL|2023-10-25|
|        26| Divya|NULL|    Delhi|        37909|2023-12-18|
|        30| Rahul|NULL|      blr|         NULL|2023-02-24|
|        43|  John|NULL|Hyderabad|         NULL|2023-10-06|
|        46| Manoj|NULL|   Mumbai|         NULL|2023-02-13|
|        47|Vikram|NULL|Bangalore|         NULL|2023-07-04|
|        49| Sneha|NULL|   Mumbai|      

In [0]:
df.filter(
    col("Age").isNotNull()
).show()

+----------+------+---+---------+-------------+----------+
|CustomerID|  Name|Age|     City|       Salary|  JoinDate|
+----------+------+---+---------+-------------+----------+
|         1| Kiran| 44|bangalore|         NULL|2023-04-01|
|         2| Meena| 24|BANGALORE|        27891|2023-09-04|
|         5|  Asha| 39|    Delhi|         NULL|2023-08-28|
|         7| Manoj|abc|    Delhi|        37696|2023-09-09|
|         8| Rahul| 21|  Chennai|         NULL|2023-04-27|
|         9| Priya|abc|bangalore|         NULL|2023-09-16|
|        10|  Ravi|abc|Bangalore|        37334|2023-04-22|
|        11|  Sana|abc|bangalore|        63409|2023-01-16|
|        12| Priya| 31|Bangalore|        63617|2023-02-18|
|        14|Vikram| 21|  Chennai|        43571|2023-10-15|
|        15| Kiran|abc|Hyderabad|         NULL|2023-05-18|
|        17| Sneha| 20|HYDERABAD|not_available|2023-06-03|
|        18|  Ravi|abc|bangalore|         NULL|2023-07-18|
|        19| Suraj|abc|    Delhi|         NULL|2023-02-1

### Count of null

In [0]:
from pyspark.sql.functions import sum,when

df.select(
    sum(when(col("Age").isNull(),1).otherwise(0)).alias("Row_Count")
).show()

+---------+
|Row_Count|
+---------+
|       53|
+---------+



### 2) Fill nulls (fillna)

In [0]:
df2 = df.fillna(
    {
        "Age":32,
        "Name":"Anonymous"
    }
)
df2.show()

+----------+------+---+---------+-------------+----------+
|CustomerID|  Name|Age|     City|       Salary|  JoinDate|
+----------+------+---+---------+-------------+----------+
|         1| Kiran| 44|bangalore|         NULL|2023-04-01|
|         2| Meena| 24|BANGALORE|        27891|2023-09-04|
|         3|  John| 32|  chennai|not_available|2023-01-04|
|         4| Meena| 32|     Pune|        40297|2023-05-18|
|         5|  Asha| 39|    Delhi|         NULL|2023-08-28|
|         6| Sneha| 32|    delhi|not_available|2023-01-10|
|         7| Manoj|abc|    Delhi|        37696|2023-09-09|
|         8| Rahul| 21|  Chennai|         NULL|2023-04-27|
|         9| Priya|abc|bangalore|         NULL|2023-09-16|
|        10|  Ravi|abc|Bangalore|        37334|2023-04-22|
|        11|  Sana|abc|bangalore|        63409|2023-01-16|
|        12| Priya| 31|Bangalore|        63617|2023-02-18|
|        13| Suraj| 32|    Delhi|         NULL|2023-06-28|
|        14|Vikram| 21|  Chennai|        43571|2023-10-1

### Fill with average

In [0]:
from pyspark.sql.functions import avg, expr

df = df.withColumn("Age", expr("try_cast(Age as int)"))

avg_age = df.select(
    avg("Age")
).collect()[0][0]

df.fillna(avg_age,subset=["Age"]).show()

+----------+------+---+---------+-------------+----------+
|CustomerID|  Name|Age|     City|       Salary|  JoinDate|
+----------+------+---+---------+-------------+----------+
|         1| Kiran| 44|bangalore|         NULL|2023-04-01|
|         2| Meena| 24|BANGALORE|        27891|2023-09-04|
|         3|  John| 32|  chennai|not_available|2023-01-04|
|         4| Meena| 32|     Pune|        40297|2023-05-18|
|         5|  Asha| 39|    Delhi|         NULL|2023-08-28|
|         6| Sneha| 32|    delhi|not_available|2023-01-10|
|         7| Manoj| 32|    Delhi|        37696|2023-09-09|
|         8| Rahul| 21|  Chennai|         NULL|2023-04-27|
|         9| Priya| 32|bangalore|         NULL|2023-09-16|
|        10|  Ravi| 32|Bangalore|        37334|2023-04-22|
|        11|  Sana| 32|bangalore|        63409|2023-01-16|
|        12| Priya| 31|Bangalore|        63617|2023-02-18|
|        13| Suraj| 32|    Delhi|         NULL|2023-06-28|
|        14|Vikram| 21|  Chennai|        43571|2023-10-1